# Section 3: Comprehensive Statistical Profiling
**Descriptive Statistics, Fault Interval Metrics & Stream Variance Profiles**

This notebook computes central tendency, dispersion, extremes, skewness, kurtosis, fault sequence durations, and nominal vs. anomalous stream variance profiles.

In [1]:
import os
import sys
import glob
import ast
import re
import pandas as pd
import numpy as np
import scipy.stats as stats
from tqdm import tqdm

# --- Path Resolution ---
candidate_base_dirs = [
    'data', 'archive/data/data', 'archive/data', '.',
    '../data', '../archive/data/data', '../archive/data', '..'
]

test_dir = None
csv_path = None

for base in candidate_base_dirs:
    te = os.path.join(base, 'test')
    if os.path.exists(te) and len(glob.glob(os.path.join(te, '*.npy'))) > 0:
        test_dir = te
        break

for base in ['archive', 'data', '.', '../archive', '../data', '..']:
    cp = os.path.join(base, 'labeled_anomalies.csv')
    if os.path.exists(cp):
        csv_path = cp
        break

df_meta = pd.read_csv(csv_path)

def parse_sequences(val):
    if isinstance(val, str):
        try:
            return ast.literal_eval(val)
        except Exception:
            pass
    return val

df_meta['anomaly_sequences'] = df_meta['anomaly_sequences'].apply(parse_sequences)

In [2]:
# --- Statistical Profiling computation ---
smap_primary_vals = []
msl_primary_vals = []
nominal_variances = []
anomalous_variances = []
anomaly_durations = []

for idx, row in tqdm(df_meta.iterrows(), total=len(df_meta), desc="Analyzing Streams"):
    chan_id = row['chan_id']
    spacecraft = row['spacecraft']
    seqs = row['anomaly_sequences']
    
    test_path = os.path.join(test_dir, f"{chan_id}.npy")
    if not os.path.exists(test_path):
        continue
        
    arr = np.load(test_path)[:, 0] # Column 0 primary sensor reading
    n_pts = len(arr)
    
    if spacecraft == 'SMAP':
        smap_primary_vals.append(arr)
    else:
        msl_primary_vals.append(arr)
        
    anom_mask = np.zeros(n_pts, dtype=bool)
    if isinstance(seqs, list):
        for start, end in seqs:
            s_idx = max(0, start)
            e_idx = min(n_pts, end)
            anom_mask[s_idx:e_idx] = True
            anomaly_durations.append(e_idx - s_idx)
            
    nom_data = arr[~anom_mask]
    anom_data = arr[anom_mask]
    
    if len(nom_data) > 0:
        nominal_variances.append(np.var(nom_data))
    if len(anom_data) > 0:
        anomalous_variances.append(np.var(anom_data))

smap_concat = np.concatenate(smap_primary_vals)
msl_concat = np.concatenate(msl_primary_vals)
all_concat = np.concatenate([smap_concat, msl_concat])

def compute_stats_profile(data_array, name):
    mode_res = stats.mode(data_array, keepdims=True)
    mode_val = float(mode_res.mode[0]) if len(mode_res.mode) > 0 else np.nan
    q25, q50, q75 = np.percentile(data_array, [25, 50, 75])
    iqr = q75 - q25
    
    return {
        "Dataset Category": name,
        "Sample Count": len(data_array),
        "Mean": float(np.mean(data_array)),
        "Median": float(np.median(data_array)),
        "Mode": mode_val,
        "Min": float(np.min(data_array)),
        "Max": float(np.max(data_array)),
        "Range": float(np.max(data_array) - np.min(data_array)),
        "Std Dev": float(np.std(data_array)),
        "Variance": float(np.var(data_array)),
        "IQR": float(iqr),
        "Skewness": float(stats.skew(data_array)),
        "Kurtosis": float(stats.kurtosis(data_array))
    }

df_stats_profile = pd.DataFrame([
    compute_stats_profile(all_concat, "Overall Continuous Telemetry"),
    compute_stats_profile(smap_concat, "SMAP Primary Telemetry (Normalized)"),
    compute_stats_profile(msl_concat, "MSL Primary Telemetry (Raw Unbounded)")
])

print("Primary Continuous Sensor Descriptive Statistics Profile:")
display(df_stats_profile.round(4))

Analyzing Streams:   0%|          | 0/82 [00:00<?, ?it/s]

Analyzing Streams:  48%|████▊     | 39/82 [00:00<00:00, 386.01it/s]

Analyzing Streams: 100%|██████████| 82/82 [00:00<00:00, 480.30it/s]

Primary Continuous Sensor Descriptive Statistics Profile:


,Dataset Category,Sample Count,Mean,Median,Mode,Min,Max,Range,Std Dev,Variance,IQR,Skewness,Kurtosis
0,Overall Continuous Telemetry,517764,-0.1409,-0.3873,-1.0,-1.4242,258.1081,259.5323,2.6057,6.7897,1.6819,46.2018,2393.5711
1,SMAP Primary Telemetry (Normalized),444035,-0.2039,-0.3892,-1.0,-1.0000,1.0000,2.0000,0.7734,0.5982,1.5996,0.4587,-1.3665
2,MSL Primary Telemetry (Raw Unbounded),73729,0.2386,-0.3107,-1.0,-1.4242,258.1081,259.5323,6.6265,43.9106,1.9053,19.5537,394.8913


In [3]:
# --- Fault Interval Duration & Stream Variance Profile ---
durations_arr = np.array(anomaly_durations)
duration_stats = {
    "Total Anomaly Instances": len(durations_arr),
    "Mean Fault Duration (timesteps)": np.mean(durations_arr),
    "Median Fault Duration (timesteps)": np.median(durations_arr),
    "Min Fault Duration": np.min(durations_arr),
    "Max Fault Duration": np.max(durations_arr),
    "Std Dev of Fault Duration": np.std(durations_arr),
    "IQR of Fault Duration": np.percentile(durations_arr, 75) - np.percentile(durations_arr, 25)
}

variance_profile = {
    "Nominal Stream Mean Variance": np.mean(nominal_variances),
    "Nominal Stream Median Variance": np.median(nominal_variances),
    "Anomalous Stream Mean Variance": np.mean(anomalous_variances),
    "Anomalous Stream Median Variance": np.median(anomalous_variances),
    "Variance Increase Ratio (Mean)": np.mean(anomalous_variances) / (np.mean(nominal_variances) + 1e-9)
}

df_durations = pd.DataFrame([duration_stats]).T.reset_index()
df_durations.columns = ["Fault Duration Metric", "Value"]

df_variances = pd.DataFrame([variance_profile]).T.reset_index()
df_variances.columns = ["Variance Profile Metric", "Value"]

print("Fault Duration Statistics:")
display(df_durations.round(2))

print("\nNominal vs. Anomalous Variance Profile:")
display(df_variances.round(4))

Fault Duration Statistics:


,Fault Duration Metric,Value
0,Total Anomaly Instances,105.00
1,Mean Fault Duration (timesteps),616.23
2,Median Fault Duration (timesteps),120.00
3,Min Fault Duration,10.00
4,Max Fault Duration,4217.00
5,Std Dev of Fault Duration,1019.06
6,IQR of Fault Duration,461.00



Nominal vs. Anomalous Variance Profile:


,Variance Profile Metric,Value
0,Nominal Stream Mean Variance,2.2997
1,Nominal Stream Median Variance,0.1106
2,Anomalous Stream Mean Variance,14.9714
3,Anomalous Stream Median Variance,0.3501
4,Variance Increase Ratio (Mean),6.5101
